In [ ]:
# ============================================================
# OpenSARShip2 Ship Type Classification Training
# 3-class version:
# Cargo / Tanker / Other Type
# ============================================================

!pip install -q torch torchvision scikit-learn tqdm

import re
import time
import shutil
import zipfile
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from tqdm import tqdm

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# 1. Unzip OpenSARShip2 to /content
# ============================================================

ZIP_PATH = Path(
    "/content/drive/MyDrive/SAR_AI_Ship_Detection/OpenSARShip2_local.zip"
)

OPEN_ROOT = Path("/content/OpenSARShip2")

if OPEN_ROOT.exists():
    shutil.rmtree(OPEN_ROOT)

print("Unzipping OpenSARShip2 to /content ...")
t0 = time.time()

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(OPEN_ROOT)

print(f"Unzip finished: {(time.time() - t0) / 60:.1f} min")

# ============================================================
# 2. Save path
# ============================================================

SAVE_DIR = Path(
    "/content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = SAVE_DIR / "opensarship2_resnet18_3class_ship_classifier.pt"

PATCH_DIR_PRIORITY = [
    "Patch_Uint8",
    "Patch_RGB",
    "Patch_cal",
    "Patch"
]

# ============================================================
# 3. Label parser
# ============================================================

def extract_raw_label(filename):
    stem = Path(filename).stem

    if stem.startswith("Visual_"):
        stem = stem.replace("Visual_", "", 1)

    stem = re.sub(
        r"_(hh|hv|vv|vh)$",
        "",
        stem,
        flags=re.IGNORECASE
    )

    m = re.match(r"(.+?)_x\d+_y\d+", stem)

    if m:
        label = m.group(1)
    else:
        label = stem.split("_")[0]

    return label.strip()


def normalize_label(raw_label):
    if raw_label == "Cargo":
        return "Cargo"

    if raw_label == "Tanker":
        return "Tanker"

    if raw_label == "Other Type":
        return "Other Type"

    return None

# ============================================================
# 4. Collect samples
# ============================================================

image_exts = [
    ".png", ".jpg", ".jpeg",
    ".tif", ".tiff", ".bmp"
]

samples = []

scene_dirs = [
    p for p in OPEN_ROOT.iterdir()
    if p.is_dir()
]

print("Scene folders:", len(scene_dirs))

for scene_dir in scene_dirs:

    selected_patch_dir = None

    for dname in PATCH_DIR_PRIORITY:
        p = scene_dir / dname

        if p.exists():
            selected_patch_dir = p
            break

    if selected_patch_dir is None:
        continue

    for img_path in selected_patch_dir.rglob("*"):

        if img_path.suffix.lower() not in image_exts:
            continue

        raw_label = extract_raw_label(img_path.name)
        label = normalize_label(raw_label)

        if label is not None:
            samples.append((img_path, label))

print("Total samples:", len(samples))

cnt = Counter([s[1] for s in samples])

print("\n7-class distribution:")
for k, v in cnt.most_common():
    print(f"{k}: {v}")

classes = [
    "Cargo",
    "Tanker",
    "Other Type"
]

class_to_idx = {
    c: i for i, c in enumerate(classes)
}

idx_to_class = {
    i: c for c, i in class_to_idx.items()
}

print("\nClasses:", classes)
print("Number of classes:", len(classes))
print("Samples:", len(samples))

# ============================================================
# 5. Train / Val split
# ============================================================

train_samples, val_samples = train_test_split(
    samples,
    test_size=0.2,
    random_state=42,
    stratify=[s[1] for s in samples]
)

print("Train:", len(train_samples))
print("Val:", len(val_samples))

# ============================================================
# 6. Dataset
# ============================================================

IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


class OpenSARShipDataset(Dataset):

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        y = class_to_idx[label]

        return img, y


train_ds = OpenSARShipDataset(train_samples, train_tf)
val_ds = OpenSARShipDataset(val_samples, val_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ============================================================
# 7. Model
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model.fc = nn.Linear(
    model.fc.in_features,
    len(classes)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ============================================================
# 8. Training
# ============================================================

epochs = 20
best_acc = 0.0
best_y_true = None
best_y_pred = None

for epoch in range(epochs):

    model.train()

    train_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{epochs}"
    )

    for imgs, labels in pbar:

        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{correct / total:.4f}"
        })

    train_loss = train_loss / total
    train_acc = correct / total

    model.eval()

    val_correct = 0
    val_total = 0

    y_true = []
    y_pred = []

    with torch.no_grad():

        for imgs, labels in tqdm(
            val_loader,
            desc="Validation"
        ):

            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(imgs)
            preds = outputs.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    val_acc = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_acc:

        best_acc = val_acc
        best_y_true = y_true
        best_y_pred = y_pred

        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": classes,
            "class_to_idx": class_to_idx,
            "idx_to_class": idx_to_class,
            "img_size": IMG_SIZE,
            "label_scheme": "7class",
            "class_names": classes
        }, MODEL_SAVE_PATH)

        print("Saved best model:", MODEL_SAVE_PATH)

# ============================================================
# 9. Final report
# ============================================================

print("Best Val Acc:", best_acc)

from sklearn.metrics import classification_report

print(
    classification_report(
        best_y_true,
        best_y_pred,
        labels=list(range(len(classes))),
        target_names=classes,
        zero_division=0
    )
)

print("Saved model:", MODEL_SAVE_PATH)

Mounted at /content/drive
Unzipping OpenSARShip2 to /content ...
Unzip finished: 0.5 min
Scene folders: 46
Total samples: 18265

7-class distribution:
Cargo: 11163
Tanker: 4025
Other Type: 3077

Classes: ['Cargo', 'Tanker', 'Other Type']
Number of classes: 3
Samples: 18265
Train: 14612
Val: 3653
Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 29.8MB/s]
Validation: 100%|██████████| 29/29 [00:09<00:00,  2.90it/s]


Epoch [1/20] Loss: 0.8414 Train Acc: 0.6249 Val Acc: 0.6411
Saved best model: /content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models/opensarship2_resnet18_3class_ship_classifier.pt


Validation: 100%|██████████| 29/29 [00:10<00:00,  2.78it/s]


Epoch [2/20] Loss: 0.7832 Train Acc: 0.6583 Val Acc: 0.6581
Saved best model: /content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models/opensarship2_resnet18_3class_ship_classifier.pt


...

Epoch [18/20] Loss: 0.5907 Train Acc: 0.7493 Val Acc: 0.6729


Validation: 100%|██████████| 29/29 [00:10<00:00,  2.77it/s]


Epoch [19/20] Loss: 0.5843 Train Acc: 0.7530 Val Acc: 0.6852


Validation: 100%|██████████| 29/29 [00:08<00:00,  3.34it/s]

Epoch [20/20] Loss: 0.5636 Train Acc: 0.7620 Val Acc: 0.6852
Best Val Acc: 0.6955926635641938
              precision    recall  f1-score   support

       Cargo       0.75      0.85      0.80      2233
      Tanker       0.56      0.45      0.50       805
  Other Type       0.57      0.47      0.51       615

    accuracy                           0.70      3653
   macro avg       0.63      0.59      0.60      3653
weighted avg       0.68      0.70      0.68      3653

Saved model: /content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models/opensarship2_resnet18_3class_ship_classifier.pt
